# MinFin — Google Colab (one-click run)

For **non-programmers**: upload your **MinFin input Excel file** (`.xlsx` or `.xlsm`), then run all cells below.

You do **not** need to upload the MinFin code package — this notebook downloads and installs it from GitHub automatically.

## How to use
1. Menu **Runtime → Run all** (or run each cell in order)
2. In the **Upload input file** cell, select your input workbook
3. Wait for the analysis to finish (may take several minutes)
4. The last cell downloads `minfin_results.zip` (Excel outputs and HTML charts)

## Input file formats
- **Legacy**: `.xlsm` with sheets such as `Definitions`
- **Pure input**: `.xlsx` with `INVESTMENT PLAN`, `TECHNOLOGY REGISTER`, etc.

See [docs/input_workbook_mapping.md](https://github.com/MINFinModel/minfin-py-public-release/blob/colab-release/docs/input_workbook_mapping.md) in the repository.

> **Maintainers**: Colab installs the code published on GitHub. Unpushed local changes will not appear in Colab; run `git push` to the branch configured below before sharing.

In [ ]:
# ── Configuration (most users can leave this unchanged) ─────────────────
REPO_URL = "https://github.com/MINFinModel/minfin-py-public-release.git"
BRANCH = "colab-release"   # synced from private MINFin-Py for Colab sharing
START_YEAR = 2025

# Set True to force a fresh clone (pull latest code from GitHub)
FORCE_RECLONE = False

# Install mode:
#   "github" — default; install MinFin from GitHub (recommended)
#   "upload" — for testing unpublished local code; upload a source zip instead
INSTALL_MODE = "github"

In [ ]:
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

REPO_DIR = Path("/content/minfin-py-public-release")
REPO_SLUG = "MINFinModel/minfin-py-public-release"
REPO_NAME = "minfin-py-public-release"


def _github_token():
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


def _download_repo_zip():
    token = _github_token()
    public_url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
    url = f"https://{token}@github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip" if token else public_url
    zip_path = Path("/content/minfin_repo.zip")

    print("Downloading:", public_url)
    try:
        urllib.request.urlretrieve(url, zip_path)
    except urllib.error.HTTPError as exc:
        if exc.code in (404, 403):
            raise RuntimeError(
                "Could not download the MinFin repository. "
                "Could not download the public MinFin repository. "
                "Check that branch colab-release exists on minfin-py-public-release."
            ) from exc
        raise

    shutil.unpack_archive(zip_path, "/content")
    zip_path.unlink(missing_ok=True)

    extracted = Path("/content") / f"{REPO_NAME}-{BRANCH}"
    if not extracted.exists():
        candidates = sorted(Path("/content").glob(f"{REPO_NAME}-*"))
        if len(candidates) == 1:
            extracted = candidates[0]
        else:
            raise FileNotFoundError(f"Unexpected archive layout after download: {candidates}")

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    extracted.rename(REPO_DIR)


if INSTALL_MODE not in ("github", "upload"):
    raise ValueError('INSTALL_MODE must be "github" or "upload"')

if INSTALL_MODE == "github":
    if FORCE_RECLONE and REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    if not REPO_DIR.exists():
        _download_repo_zip()
    else:
        print(f"Using existing repo at {REPO_DIR} (set FORCE_RECLONE=True to refresh)")

elif INSTALL_MODE == "upload":
    from google.colab import files

    print("Upload a MinFin source zip (must contain setup.py and the MinFin/ folder)")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("In upload mode, please upload exactly one zip file")
    zip_name = next(iter(uploaded))
    if FORCE_RECLONE and REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.unpack_archive(zip_name, "/content", format="zip")
    # If the zip root folder name differs, find the directory that contains setup.py
    if not (REPO_DIR / "setup.py").exists():
        candidates = [p for p in Path("/content").iterdir() if (p / "setup.py").exists()]
        if len(candidates) == 1:
            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            candidates[0].rename(REPO_DIR)
        else:
            raise FileNotFoundError(
                "The zip must contain setup.py; zip the full repository root and try again"
            )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nbconvert"])

import MinFin

print("MinFin installed from:", Path(MinFin.__file__).resolve())

In [ ]:
from google.colab import files
import os

print("Upload your MinFin input Excel file (.xlsx or .xlsm)")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please upload exactly one input file")

file_path = os.path.abspath(next(iter(uploaded)))
suffix = os.path.splitext(file_path)[1].lower()
if suffix not in (".xlsx", ".xlsm"):
    raise ValueError(f"Unsupported file type: {suffix}. Please upload .xlsx or .xlsm")

print("Input file:", file_path)
print("START_YEAR:", START_YEAR)

In [ ]:
import re
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor
from pathlib import Path

repo_dir = Path("/content/minfin-py-public-release")
nb_path = repo_dir / "minfin_notebook.ipynb"
if not nb_path.exists():
    raise FileNotFoundError(
        f"Could not find {nb_path}. Check that the GitHub branch includes minfin_notebook.ipynb"
    )

nb = nbformat.read(nb_path.open(encoding="utf-8"), as_version=4)

escaped_file_path = file_path.replace("\\", "\\\\")
patched = False
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    new_src = re.sub(
        r'^(?!\s*#)(\s*)file_path\s*=\s*["\'].*?["\']',
        rf'\1file_path = r"{escaped_file_path}"',
        src,
        flags=re.MULTILINE,
    )
    new_src = re.sub(
        r'^(?!\s*#)(\s*)START_YEAR\s*=\s*\d+',
        rf'\1START_YEAR = {START_YEAR}',
        new_src,
        flags=re.MULTILINE,
    )
    if new_src != src:
        cell.source = new_src
        patched = True

if not patched:
    raise RuntimeError("Could not locate file_path / START_YEAR assignments in minfin_notebook.ipynb")

print("Running minfin_notebook.ipynb — this may take several minutes, please wait…")
ep = ExecutePreprocessor(timeout=7200, kernel_name="python3")
ep.preprocess(nb, {"metadata": {"path": str(repo_dir)}})

executed_path = repo_dir / "minfin_notebook_executed.ipynb"
nbformat.write(nb, executed_path.open("w", encoding="utf-8"))
print("Analysis complete. Executed notebook saved to:", executed_path)

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

output_dir = Path("/content/minfin-py-public-release/minfin_output")
if not output_dir.exists():
    raise FileNotFoundError(
        f"Output directory not found: {output_dir}. Check that the cells above ran successfully."
    )

zip_base = "/content/minfin_results"
if Path(zip_base + ".zip").exists():
    Path(zip_base + ".zip").unlink()

archive = shutil.make_archive(zip_base, "zip", output_dir)
print("Archive created:", archive)
print("Output files:")
for p in sorted(output_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(output_dir))

files.download(archive)